In [1]:
import pandas as pd
from pathlib import Path

HI_ACCOUNT_DATA_PATH = Path().cwd() / "Data-HI" / "HI-Small_accounts.csv"
HI_TRANSACTIONS_DATA_PATH = Path().cwd() / "Data-HI" / "HI-Small_Trans.csv"

accounts_df = pd.read_csv(HI_ACCOUNT_DATA_PATH)
trans_df = pd.read_csv(HI_TRANSACTIONS_DATA_PATH)

In [2]:
trans_df['Timestamp'] = pd.to_datetime(trans_df['Timestamp'])

trans_df = trans_df.rename(columns={'Account':'From Account'})
trans_df = trans_df.rename(columns={'Account.1':'To Account'})

trans_df['From Bank'] = trans_df['From Bank'].astype('category')
trans_df['To Bank'] = trans_df['To Bank'].astype('category')
trans_df['Payment Currency'] = trans_df['Payment Currency'].astype('category')
trans_df['Receiving Currency'] = trans_df['Receiving Currency'].astype('category')
trans_df['Payment Format'] = trans_df['Payment Format'].astype('category')

In [3]:
trans_df.dtypes

Timestamp             datetime64[us]
From Bank                   category
From Account                     str
To Bank                     category
To Account                       str
Amount Received              float64
Receiving Currency          category
Amount Paid                  float64
Payment Currency            category
Payment Format              category
Is Laundering                  int64
dtype: object

In [5]:
accounts_df = accounts_df.rename(columns={'Account Number': 'Account'})

In [6]:
accounts_df.dtypes

Bank Name        str
Bank ID        int64
Account          str
Entity ID        str
Entity Name      str
dtype: object

In [7]:
accounts_df['Universal_Account_ID'] = accounts_df['Bank ID'].astype(str) + "_" + accounts_df['Account'].astype(str)

trans_df['From_Universal_ID'] = trans_df['From Bank'].astype(str) + "_" + trans_df['From Account'].astype(str)
trans_df['To_Universal_ID'] = trans_df['To Bank'].astype(str) + "_" + trans_df['To Account'].astype(str)

In [21]:
trans_df = trans_df.sort_values('Timestamp').reset_index(drop=True)
trans_df.index.name = 'transaction_id'

In [22]:
entity_map = accounts_df.set_index('Universal_Account_ID')['Entity ID'].to_dict()
trans_df['Entity ID'] = trans_df['From_Universal_ID'].map(entity_map)

trans_df = trans_df.sort_values('Timestamp').reset_index(drop=True)
trans_df['transaction_id'] = trans_df.index
temp_df = trans_df.set_index('Timestamp')

In [23]:
roll_acc_24h = temp_df.groupby('From_Universal_ID').rolling('24h').agg(
    acc_vol_24h=('Amount Paid', 'sum'),
    acc_count_24h=('Amount Paid', 'count')
).reset_index()

In [24]:
roll_acc_7d = temp_df.groupby('From_Universal_ID').rolling('168h').agg(
    acc_vol_7d=('Amount Paid', 'sum')
).reset_index()

In [25]:
roll_ent = temp_df.groupby('Entity ID').rolling('24h').agg(
    ent_vol_24h=('Amount Paid', 'sum'),
    ent_count_24h=('Amount Paid', 'count')
).reset_index()

In [26]:
roll_fmt = temp_df.groupby(['From_Universal_ID', 'Payment Format']).rolling('24h').agg(
    acc_fmt_vol_24h=('Amount Paid', 'sum')
).reset_index()

In [27]:
trans_df = trans_df.merge(roll_acc_24h[['transaction_id', 'acc_vol_24h', 'acc_count_24h']], on='tx_id', how='left')
trans_df = trans_df.merge(roll_acc_7d[['transaction_id', 'acc_vol_7d']], on='tx_id', how='left')
trans_df = trans_df.merge(roll_ent[['transaction_id', 'ent_vol_24h', 'ent_count_24h']], on='tx_id', how='left')
trans_df = trans_df.merge(roll_fmt[['transaction_id', 'acc_fmt_vol_24h']], on='tx_id', how='left')

KeyError: "['transaction_id'] not in index"